# 비율 파생을 NN 입력에 — 임베딩-MLP (비율 X vs 비율 O) + 블렌드 증분

**가설:** NN은 비율을 근사할 수 있지만 정형데이터에선 **명시 비율 피처가 도움.** 게이팅 비율의 NaN은 **결측플래그**가 잡아 NN에 적합.
**검증(비용 절약):** ① **Phase 0**(서브샘플) 비율X vs 비율O val AUC — 비율이 NN에 도움되나 *싸게* 먼저. ② 도움되면 풀 5-fold 둘 다 → 단독 AUC·ρ. ③ **블렌드 증분**(트리+lr에 NN_noratio vs NN_ratio).
**★ GPU 세션.** 입력: train/test + `oof_predictions.csv`(트리+lr) + `prune_list.json`(keep, 없으면 자동).
비율은 num 블록에 주입(스케일+결측플래그) — 기존 keep 피처와 동일 처리.

## 1. 셋업 + GPU

In [1]:
import re,os,glob,json,warnings,time
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
warnings.filterwarnings("ignore")
DEVICE="cuda" if torch.cuda.is_available() else "cpu"; print("torch",torch.__version__,"| device:",DEVICE)
if DEVICE=="cpu": print("⚠️ GPU 미감지 — Settings>Accelerator>GPU 켜고 재실행.")
def runr(p): return rankdata(p)/len(p)

torch 2.10.0+cu128 | device: cuda


## 2. 데이터 + 설정

In [2]:
csvs=sorted(glob.glob("/kaggle/input/**/*.csv",recursive=True))
def pick(*ks):
    for p in csvs:
        if all(k in os.path.basename(p).lower() for k in ks): return p
train=pd.read_csv(pick("train") or csvs[0]); test=pd.read_csv(pick("test"))
TARGET="임신 성공 여부"; ID_COL="ID"; y=train[TARGET].astype(int).values
SEED=42; N_SPLITS=5
PROTO_N=40000; PROTO_EPOCHS=8
EPOCHS=40; PATIENCE=6; BATCH=2048; LR=1e-3; HIDDEN=(256,128); DROPOUT=0.2; BOOT_N=2000
kp=glob.glob("/kaggle/input/**/prune_list.json",recursive=True)+glob.glob("prune_list.json")
KEEP=json.load(open(kp[0]))["keep"] if kp else None
print("train",train.shape,"| keep",len(KEEP) if KEEP else "자동산출")

train (256351, 69) | keep 34


## 3. 입력 파이프라인 + 비율 주입 (fold-내부·누수안전)

In [3]:
NOMINAL_COLS=["시술 시기 코드","시술 유형","배란 유도 유형","난자 출처","정자 출처"]
ORDINAL_COUNT_COLS=["총 시술 횟수","클리닉 내 총 시술 횟수","IVF 시술 횟수","DI 시술 횟수","총 임신 횟수","IVF 임신 횟수","DI 임신 횟수","총 출산 횟수","IVF 출산 횟수","DI 출산 횟수"]
AGE_MAPS={"시술 당시 나이":{"만18-34세":0,"만35-37세":1,"만38-39세":2,"만40-42세":3,"만43-44세":4,"만45-50세":5,"알 수 없음":-1},
 "난자 기증자 나이":{"만20세 이하":0,"만21-25세":1,"만26-30세":2,"만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1},
 "정자 기증자 나이":{"만20세 이하":0,"만21-25세":1,"만26-30세":2,"만31-35세":3,"만36-40세":4,"만41-45세":5,"알 수 없음":-1}}
COUNT_MAP={"0회":0,"1회":1,"2회":2,"3회":3,"4회":4,"5회":5,"6회 이상":6}
def NUM(df,c): return pd.to_numeric(df[c],errors="coerce") if c in df else pd.Series(np.nan,index=df.index)
def DIV(num,den): den=den.astype(float); return num.astype(float)/den.where(den>0)
def masks(df): return {"신선":NUM(df,"신선 배아 사용 여부")==1,"동결":NUM(df,"동결 배아 사용 여부")==1,"ICSI":NUM(df,"미세주입된 난자 수")>0,"본인난자":df["난자 출처"].astype(str)=="본인 제공","기증난자":df["난자 출처"].astype(str)=="기증 제공"}
def build_ratios(df):
    M=masks(df); F={}
    P1=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"혼합된 난자 수")); P2=DIV(NUM(df,"미세주입에서 생성된 배아 수"),NUM(df,"미세주입된 난자 수"))
    P6=DIV(NUM(df,"총 생성 배아 수"),NUM(df,"수집된 신선 난자 수")); L3=NUM(df,"배아 이식 경과일")-NUM(df,"난자 혼합 경과일")
    F["r_P1"]=P1; F["r_P2"]=P2; F["r_P6"]=P6; F["r_L3"]=L3
    F["r_P3"]=DIV(NUM(df,"이식된 배아 수")+NUM(df,"저장된 배아 수"),NUM(df,"총 생성 배아 수")); F["r_P5"]=DIV(NUM(df,"저장된 배아 수"),NUM(df,"총 생성 배아 수"))
    F["r_g신선수정"]=P1.where(M["신선"]); F["r_gICSI"]=P2.where(M["ICSI"]); F["r_g본인수율"]=P6.where(M["본인난자"]); F["r_g신선배양"]=L3.where(M["신선"])
    F["r_FZ2"]=DIV(NUM(df,"이식된 배아 수"),NUM(df,"해동된 배아 수"))
    return pd.DataFrame(F,index=df.index)
def prep_global(df):
    df=df.copy()
    for c in ORDINAL_COUNT_COLS:
        if c in df: df[c]=df[c].map(COUNT_MAP)
    for c,m in AGE_MAPS.items():
        if c in df: df[c]=df[c].map(m)
    return df
RATc=list(build_ratios(train.head(2)).columns)
trg=pd.concat([prep_global(train),build_ratios(train)],axis=1); teg=pd.concat([prep_global(test),build_ratios(test)],axis=1)
if KEEP is None:
    print("prune_list.json 없음 → 순열중요도 keep 산출(LGBM, ~수분)...")
    import lightgbm as lgb
    Xp=train.drop(columns=[TARGET,ID_COL]).copy()
    for c in Xp.columns:
        if not pd.api.types.is_numeric_dtype(Xp[c]): Xp[c]=pd.factorize(Xp[c].astype(str))[0]
    Xp=Xp.apply(pd.to_numeric,errors="coerce"); fp=list(Xp.columns)
    a,b,ya,yb=train_test_split(Xp,y,test_size=0.25,stratify=y,random_state=SEED)
    mp=lgb.train(dict(objective="binary",metric="auc",learning_rate=0.05,num_leaves=63,verbose=-1,seed=SEED),lgb.Dataset(a,ya),500,valid_sets=[lgb.Dataset(b,yb)],callbacks=[lgb.early_stopping(50,verbose=False),lgb.log_evaluation(0)])
    bs=roc_auc_score(yb,mp.predict(b)); rp=np.random.default_rng(0); dr={}
    for c in fp:
        Xq=b.copy(); Xq[c]=Xq[c].values[rp.permutation(len(Xq))]; dr[c]=bs-roc_auc_score(yb,mp.predict(Xq))
    KEEP=[c for c in fp if dr[c]>=0.0001]; json.dump({"keep":KEEP},open("prune_list.json","w"),ensure_ascii=False)
def nn_cols(tr,keep,with_ratios):
    keep=[c for c in keep if c in tr.columns and c not in (TARGET,ID_COL)]
    cat=[c for c in keep if not pd.api.types.is_numeric_dtype(tr[c])]; num=[c for c in keep if c not in cat]
    if with_ratios: num=num+RATc          # 비율을 num 블록에 (스케일+결측플래그)
    return cat,num
def fit_fold(trdf,cat,num):
    enc={c:{v:i+1 for i,v in enumerate(trdf[c].astype(str).fillna("NA").value_counts().index)} for c in cat}
    med={c:float(trdf[c].median()) for c in num}; mu={}; sd={}
    for c in num:
        x=trdf[c].fillna(med[c]).astype(float); mu[c]=float(x.mean()); sd[c]=float(x.std())+1e-6
    return enc,med,mu,sd
def transform(df,cat,num,enc,med,mu,sd):
    Xc=np.zeros((len(df),len(cat)),dtype=np.int64)
    for j,c in enumerate(cat): Xc[:,j]=df[c].astype(str).fillna("NA").map(enc[c]).fillna(0).astype(int).values
    Xn=np.zeros((len(df),len(num)*2),dtype=np.float32)
    for j,c in enumerate(num):
        Xn[:,2*j+1]=df[c].isna().astype(np.float32).values
        x=df[c].fillna(med[c]).astype(float).values; Xn[:,2*j]=((x-mu[c])/sd[c]).astype(np.float32)
    return Xc,Xn
print("비율",len(RATc),"개 num에 주입 | keep",len(KEEP))

비율 11 개 num에 주입 | keep 34


## 4. 모델 (임베딩-MLP)

In [4]:
class EmbMLP(nn.Module):
    def __init__(self,cat_dims,n_num,hidden=(256,128),p=0.2):
        super().__init__()
        self.embs=nn.ModuleList([nn.Embedding(d,min(50,(d+1)//2+1)) for d in cat_dims])
        et=sum(e.embedding_dim for e in self.embs); layers=[]; din=et+n_num
        for h in hidden: layers+=[nn.Linear(din,h),nn.BatchNorm1d(h),nn.ReLU(),nn.Dropout(p)]; din=h
        layers+=[nn.Linear(din,1)]; self.mlp=nn.Sequential(*layers)
    def forward(self,xc,xn):
        if len(self.embs)>0: x=torch.cat([torch.cat([emb(xc[:,i]) for i,emb in enumerate(self.embs)],1),xn],1)
        else: x=xn
        return self.mlp(x).squeeze(1)
def train_nn(Xc_t,Xn_t,yt,Xc_v,Xn_v,yv,dims,epochs,patience,bs,lr,hidden,dropout,seed=42):
    torch.manual_seed(seed); np.random.seed(seed)
    if DEVICE=="cuda": torch.cuda.manual_seed_all(seed)
    model=EmbMLP(dims,Xn_t.shape[1],hidden,dropout).to(DEVICE); opt=torch.optim.Adam(model.parameters(),lr=lr,weight_decay=1e-5); lossf=nn.BCEWithLogitsLoss()
    dl=DataLoader(TensorDataset(torch.as_tensor(Xc_t),torch.as_tensor(Xn_t),torch.as_tensor(yt,dtype=torch.float32)),batch_size=bs,shuffle=True,drop_last=True)
    Xcv=torch.as_tensor(Xc_v).to(DEVICE); Xnv=torch.as_tensor(Xn_v).to(DEVICE); best=-1; bstate=None; bad=0
    for ep in range(epochs):
        model.train()
        for xc,xn,yb in dl:
            xc,xn,yb=xc.to(DEVICE),xn.to(DEVICE),yb.to(DEVICE); opt.zero_grad(); loss=lossf(model(xc,xn),yb); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad(): auc=roc_auc_score(yv,torch.sigmoid(model(Xcv,Xnv)).cpu().numpy())
        if auc>best+1e-5: best=auc; bstate={k:v.cpu().clone() for k,v in model.state_dict().items()}; bad=0
        else:
            bad+=1
            if bad>=patience: break
    model.load_state_dict(bstate); return model,best
def predict_nn(model,Xc,Xn,bs=16384):
    model.eval(); out=[]
    with torch.no_grad():
        for i in range(0,len(Xc),bs):
            out.append(torch.sigmoid(model(torch.as_tensor(Xc[i:i+bs]).to(DEVICE),torch.as_tensor(Xn[i:i+bs]).to(DEVICE))).cpu().numpy())
    return np.concatenate(out)

## 5. ★ Phase 0 — 비율X vs 비율O 프로토타입 (싸게 비교)

In [5]:
def proto(with_ratios):
    cat,num=nn_cols(trg,KEEP,with_ratios)
    idx=np.arange(len(trg)); sub,_=train_test_split(idx,train_size=min(PROTO_N,len(trg)),stratify=y,random_state=SEED)
    sd_=trg.iloc[sub].reset_index(drop=True); ys=y[sub]
    tri,vai=train_test_split(np.arange(len(sd_)),test_size=0.2,stratify=ys,random_state=SEED)
    enc,med,mu,sd=fit_fold(sd_.iloc[tri],cat,num)
    Xc_t,Xn_t=transform(sd_.iloc[tri],cat,num,enc,med,mu,sd); Xc_v,Xn_v=transform(sd_.iloc[vai],cat,num,enc,med,mu,sd)
    _,a=train_nn(Xc_t,Xn_t,ys[tri],Xc_v,Xn_v,ys[vai],[len(enc[c])+1 for c in cat],PROTO_EPOCHS,3,BATCH,LR,HIDDEN,DROPOUT)
    return a
t0=time.time(); a_no=proto(False); a_ra=proto(True)
print(f"[Phase 0] 비율X val AUC={a_no:.5f} | 비율O val AUC={a_ra:.5f} | Δ={a_ra-a_no:+.5f}  ({time.time()-t0:.0f}s)")
RATIO_HELPS = a_ra > a_no + 0.0005   # 프로토타입서 비율이 도움 신호
print("→ 비율 도움 신호:", "✅ 풀런 진행" if RATIO_HELPS else "⚠️ 미약 — 풀런 전 재고(비용 절약). 그래도 풀런 강제하려면 아래 FORCE=True")
FORCE=False

[Phase 0] 비율X val AUC=0.73071 | 비율O val AUC=0.73303 | Δ=+0.00232  (10s)
→ 비율 도움 신호: ✅ 풀런 진행


## 6. Phase 1 — 풀 5-fold (비율X·비율O) + 단독 AUC·ρ

In [6]:
oof_no=oof_ra=None
if RATIO_HELPS or FORCE:
    folds=list(StratifiedKFold(N_SPLITS,shuffle=True,random_state=SEED).split(trg,y))
    def full(with_ratios):
        cat,num=nn_cols(trg,KEEP,with_ratios); oof=np.zeros(len(trg))
        for k,(tri,vai) in enumerate(folds):
            enc,med,mu,sd=fit_fold(trg.iloc[tri],cat,num)
            Xc_t,Xn_t=transform(trg.iloc[tri],cat,num,enc,med,mu,sd); Xc_v,Xn_v=transform(trg.iloc[vai],cat,num,enc,med,mu,sd)
            m,va=train_nn(Xc_t,Xn_t,y[tri],Xc_v,Xn_v,y[vai],[len(enc[c])+1 for c in cat],EPOCHS,PATIENCE,BATCH,LR,HIDDEN,DROPOUT,SEED)
            oof[vai]=predict_nn(m,Xc_v,Xn_v); print(f"    fold{k} val={va:.5f}")
        return oof
    print("비율X 풀런:"); oof_no=full(False); print("비율O 풀런:"); oof_ra=full(True)
    print(f"\n단독 OOF — 비율X={roc_auc_score(y,oof_no):.5f} | 비율O={roc_auc_score(y,oof_ra):.5f} | Δ={roc_auc_score(y,oof_ra)-roc_auc_score(y,oof_no):+.5f}")
    print("ρ(비율X,비율O):",round(np.corrcoef(runr(oof_no),runr(oof_ra))[0,1],3))
else: print("Phase 0서 비율 도움 미약 → 풀런 생략. (FORCE=True로 강제 가능)")

비율X 풀런:
    fold0 val=0.73511
    fold1 val=0.74062
    fold2 val=0.73849
    fold3 val=0.73509
    fold4 val=0.73910
비율O 풀런:
    fold0 val=0.73556
    fold1 val=0.74126
    fold2 val=0.73915
    fold3 val=0.73616
    fold4 val=0.73810

단독 OOF — 비율X=0.73751 | 비율O=0.73797 | Δ=+0.00046
ρ(비율X,비율O): 0.982


## 7. 블렌드 증분 — 트리+lr에 NN(비율X) vs NN(비율O)

In [7]:
if oof_ra is not None:
    od=None
    for p in glob.glob("/kaggle/input/**/oof_predictions.csv",recursive=True): od=pd.read_csv(p); break
    assert od is not None, "oof_predictions.csv 필요(Add Input)"
    mem={}
    for k in od.columns:
        kl=k.lower()
        if kl in ("oof_lgb","oof_cat","oof_xgb"): mem[kl.replace("oof_","")]=runr(od[k].values)
        if kl=="oof_lr" or (("lr" in kl) and kl!="y"): mem["lr"]=runr(od[k].values)
    assert len(mem)>=3, "트리/lr OOF 부족"
    def hill(d,yy,n=120):
        nm=list(d); s0={k:roc_auc_score(yy,d[k]) for k in nm}; b=max(s0,key=s0.get); ens=[b]; s=d[b].copy(); best=(list(ens),s0[b])
        for _ in range(n):
            cb,ca=None,-1
            for k in nm:
                a=roc_auc_score(yy,(s+d[k])/(len(ens)+1))
                if a>ca: ca,cb=a,k
            ens.append(cb); s=s+d[cb]
            if ca>best[1]: best=(list(ens),ca)
        from collections import Counter; c=Counter(best[0]); return {k:c.get(k,0)/len(best[0]) for k in nm},best[1]
    dB={**mem,"nn":runr(oof_no)}; dF={**mem,"nn":runr(oof_ra)}
    wB,aB=hill(dB,y); wF,aF=hill(dF,y)
    print(f"블렌드 (NN 비율X) OOF={aB:.5f} nn가중={wB.get('nn',0):.3f}")
    print(f"블렌드 (NN 비율O) OOF={aF:.5f} nn가중={wF.get('nn',0):.3f}")
    pB=sum(wB[k]*dB[k] for k in wB); pF=sum(wF[k]*dF[k] for k in wF)
    rng=np.random.default_rng(0); ds=[roc_auc_score(y[ix],pF[ix])-roc_auc_score(y[ix],pB[ix]) for ix in (rng.integers(0,len(y),len(y)) for _ in range(BOOT_N))]
    lo,hi=np.percentile(ds,[2.5,97.5])
    print(f"\n블렌드 증분(비율효과) = {aF-aB:+.5f} | 95%CI [{lo:+.5f},{hi:+.5f}] →","✅ 비율이 NN 통해 블렌드 기여" if lo>0 else "❌ 미달")
    rho=max(np.corrcoef(runr(oof_ra),mem[k])[0,1] for k in mem if k!='lr')
    print("NN(비율O) 트리 최대ρ:",round(rho,3),"(<0.93 decorrelated)")
    pd.DataFrame({"oof_nn":oof_ra,"y":y}).to_csv("oof_nn_ratio.csv",index=False)
    json.dump({"proto_no":a_no,"proto_ra":a_ra,"oof_no":float(roc_auc_score(y,oof_no)),"oof_ra":float(roc_auc_score(y,oof_ra)),"blend_inc":float(aF-aB),"ci":[lo,hi]},open("nn_ratio_decision.json","w"),ensure_ascii=False)
    print("💾 oof_nn_ratio.csv / nn_ratio_decision.json 저장")
else: print("풀런 없음 → 블렌드 증분 생략.")

블렌드 (NN 비율X) OOF=0.74045 nn가중=0.237
블렌드 (NN 비율O) OOF=0.74049 nn가중=0.265

블렌드 증분(비율효과) = +0.00004 | 95%CI [-0.00009,+0.00016] → ❌ 미달
NN(비율O) 트리 최대ρ: 0.975 (<0.93 decorrelated)
💾 oof_nn_ratio.csv / nn_ratio_decision.json 저장


## 8. 결정
- **Phase 0 비율O > 비율X +0.0005** → 비율이 NN에 도움 신호 → 풀런.
- **블렌드 증분 CI>0 AND ρ≤0.93** → 비율 넣은 NN 채택 후보 → 멀티시드+LB.
- **CI 0 포함** → 비율이 NN 단독은 도와도 블렌드 전이 안 됨(공정 네거티브). 선형 결과와 함께 "비율의 블렌드 가치" 종합 판정.

# 파생 조합 탐색 + 최적 제출 (커밋된 OOF/test 로드 전용 · 재학습 X)

기존에 돌려 **커밋해둔** OOF/test를 다 불러서, **모델별 파생 ON/OFF 조합**을 그리드로 보고 **최적 조합으로 제출.** (재학습 없음 ~1분.)

**Add Input (커밋된 데이터셋들):**
- `oof_predictions.csv`+`test_predictions.csv` (base 트리 lgb/cat/xgb + 선형 lr)
- `oof_v2_trees.csv`+`test_v2_trees.csv` (v2게이팅 트리)
- `oof_lin_ratio.csv`+`test_lin_ratio.csv` (비율 선형)
- `oof_nn.csv`+`test_nn.csv` (NN base) · `oof_nn_ratio.csv` (비율 NN, 표용)
- train.csv (y) + sample_submission.csv
*없는 파일은 자동 skip — 있는 멤버만으로 그리드.*

In [13]:
import glob, os
print("=== /kaggle/input 의 모든 CSV ===")
for p in sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True)):
    print(os.path.relpath(p, "/kaggle/input"))

=== /kaggle/input 의 모든 CSV ===
datasets/jaeranlee/subfertility-input/sample_submission.csv
datasets/jaeranlee/subfertility-input/test.csv
datasets/jaeranlee/subfertility-input/train.csv
notebooks/jaeranlee/subfertility-predict-ai-3tree-lr/oof_predictions.csv
notebooks/jaeranlee/subfertility-predict-ai-3tree-lr/submission_blend4.csv
notebooks/jaeranlee/subfertility-predict-ai-3tree-lr/submission_lr.csv
notebooks/jaeranlee/subfertility-predict-ai-5model-fe/oof_v2_trees.csv
notebooks/jaeranlee/subfertility-predict-ai-5model-fe/test_v2_trees.csv
notebooks/jaeranlee/subfertility-predict-ai-lr-fe-0622/oof_lin_ratio.csv
notebooks/jaeranlee/subfertility-predict-ai-lr-fe-0622/test_lin_ratio.csv
notebooks/jaeranlee/subfertility-predict-ai-mlp/oof_nn.csv
notebooks/jaeranlee/subfertility-predict-ai-mlp/submission_nn_blend.csv
notebooks/jaeranlee/subfertility-predict-ai-mlp/test_nn.csv


## 0. 로드 — 커밋된 OOF/test 자동 매핑

In [12]:
import os,glob,json
import numpy as np, pandas as pd
from itertools import product
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
def find(name):
    for base in ["/kaggle/working/","/kaggle/input/","./"]:
        h=[p for p in glob.glob(f"{base}**/{name}",recursive=True) if os.path.basename(p)==name]
        if h: return sorted(h,key=len)[0]
    return None
train=pd.read_csv(find("train.csv")); TARGET="임신 성공 여부"; ID_COL="ID"; y=train[TARGET].astype(int).values; N=len(train)
def runr(x): return rankdata(x)/len(x)
# (파일, {컬럼: 멤버키})
OOF_MAP=[("oof_predictions.csv",{"oof_lgb":"lgb_base","oof_cat":"cat_base","oof_xgb":"xgb_base","oof_lr":"lin_base"}),
         ("oof_v2_trees.csv",{"v2_lgb":"lgb_v2","v2_cat":"cat_v2","v2_xgb":"xgb_v2"}),
         ("oof_lin_ratio.csv",{"oof_lin_ratio":"lin_ratio"}),
         ("oof_nn.csv",{"oof_nn":"nn_base"}),("oof_nn_ratio.csv",{"oof_nn":"nn_ratio"})]
TST_MAP=[("test_predictions.csv",{"test_lgb":"lgb_base","test_cat":"cat_base","test_xgb":"xgb_base","test_lr":"lin_base"}),
         ("test_v2_trees.csv",{"v2_lgb":"lgb_v2","v2_cat":"cat_v2","v2_xgb":"xgb_v2"}),
         ("test_lin_ratio.csv",{"lin_ratio":"lin_ratio"}),("test_nn.csv",{"test_nn":"nn_base"})]
OOF={}; TST={}; align_warn=[]
for fn,cmap in OOF_MAP:
    p=find(fn)
    if not p: print(f"  [OOF 없음] {fn}"); continue
    d=pd.read_csv(p)
    if len(d)!=N: print(f"  ⚠️ {fn} 길이 {len(d)}≠{N} — skip"); continue
    if "y" in d.columns and not np.array_equal(d["y"].astype(int).values,y): align_warn.append(fn)  # fold/정렬 점검
    got=[]
    for col,key in cmap.items():
        if col in d.columns: OOF[key]=d[col].values; got.append(key)
    print(f"  [OOF] {fn} → {got}")
for fn,cmap in TST_MAP:
    p=find(fn)
    if not p: print(f"  [test 없음] {fn}"); continue
    d=pd.read_csv(p)
    for col,key in cmap.items():
        if col in d.columns: TST[key]=d[col].values
print("\n로드 OOF 멤버:",sorted(OOF)); print("로드 test 멤버:",sorted(TST))
if align_warn: print("🔴 정렬 경고(저장 y≠train y):",align_warn," — fold 정렬 깨졌을 수 있음, 결과 신뢰 전 확인!")

  [OOF] oof_predictions.csv → ['lgb_base', 'cat_base', 'xgb_base', 'lin_base']
  [OOF] oof_v2_trees.csv → ['lgb_v2', 'cat_v2', 'xgb_v2']
  [OOF] oof_lin_ratio.csv → ['lin_ratio']
  [OOF] oof_nn.csv → ['nn_base']
  [OOF] oof_nn_ratio.csv → ['nn_ratio']
  [test 없음] test_predictions.csv

로드 OOF 멤버: ['cat_base', 'cat_v2', 'lgb_base', 'lgb_v2', 'lin_base', 'lin_ratio', 'nn_base', 'nn_ratio', 'xgb_base', 'xgb_v2']
로드 test 멤버: ['cat_v2', 'lgb_v2', 'lin_ratio', 'nn_base', 'xgb_v2']


## 1. 정렬 sanity — base 트리 블렌드가 ~0.74 나오나

In [14]:
def hill(d,yy,n=120):
    nm=list(d); s0={k:roc_auc_score(yy,d[k]) for k in nm}; b=max(s0,key=s0.get); ens=[b]; s=d[b].copy(); best=(list(ens),s0[b])
    for _ in range(n):
        cb,ca=None,-1
        for k in nm:
            a=roc_auc_score(yy,(s+d[k])/(len(ens)+1))
            if a>ca: ca,cb=a,k
        ens.append(cb); s=s+d[cb]
        if ca>best[1]: best=(list(ens),ca)
    from collections import Counter; c=Counter(best[0]); return {k:c.get(k,0)/len(best[0]) for k in nm}, best[1]
need=["lgb_base","cat_base","xgb_base"]
assert all(k in OOF for k in need), "base 트리 OOF 필수(oof_predictions.csv)"
base3={k.split("_")[0]:runr(OOF[k]) for k in need}
_,a3=hill(base3,y)
print(f"base 3트리 블렌드 OOF = {a3:.5f}  (~0.740 이어야 정렬 정상)")
if a3<0.735: print("🔴 0.735 미만 — fold 정렬 의심! 같은 seed/fold로 생성된 OOF인지 확인.")
else: print("✅ 정렬 정상 — 블렌드 신뢰 가능.")

base 3트리 블렌드 OOF = 0.74012  (~0.740 이어야 정렬 정상)
✅ 정렬 정상 — 블렌드 신뢰 가능.


## 2. §A 단독 AUC 표 — 멤버별 base vs 파생

In [15]:
rows=[]
for m,(b,d) in {"LGBM":("lgb_base","lgb_v2"),"XGBoost":("xgb_base","xgb_v2"),"CatBoost":("cat_base","cat_v2"),"선형":("lin_base","lin_ratio"),"NN":("nn_base","nn_ratio")}.items():
    ab=roc_auc_score(y,OOF[b]) if b in OOF else None; ad=roc_auc_score(y,OOF[d]) if d in OOF else None
    rows.append({"모델":m,"base":round(ab,5) if ab else "—","파생":round(ad,5) if ad else "—","Δ":round(ad-ab,5) if (ab and ad) else "—"})
print(pd.DataFrame(rows).to_string(index=False))

      모델    base      파생        Δ
    LGBM 0.73944 0.73959  0.00015
 XGBoost 0.73959 0.73957 -0.00001
CatBoost 0.73963 0.73970  0.00007
      선형 0.71904 0.72029  0.00124
      NN 0.73751 0.73797  0.00046


## 3. ★§B 조합 그리드 — 트리 각 base/v2 × 선형 base/비율 (nn 고정)

In [16]:
def vers(m,alt): return [v for v in ["base",alt] if f"{m}_{v}" in OOF]
LGV,XGV,CTV,LNV=vers("lgb","v2"),vers("xgb","v2"),vers("cat","v2"),vers("lin","ratio")
NNK="nn_base" if "nn_base" in OOF else None
grid=[]
for lg,xg,ct,ln in product(LGV,XGV,CTV,LNV):
    d={"lgb":runr(OOF[f"lgb_{lg}"]),"xgb":runr(OOF[f"xgb_{xg}"]),"cat":runr(OOF[f"cat_{ct}"]),"lin":runr(OOF[f"lin_{ln}"])}
    if NNK: d["nn"]=runr(OOF[NNK])
    w,a=hill(d,y); grid.append({"lgb":lg,"xgb":xg,"cat":ct,"lin":ln,"OOF":round(a,5),"w":{k:round(v,2) for k,v in w.items() if v>0}})
gdf=pd.DataFrame(grid).sort_values("OOF",ascending=False).reset_index(drop=True)
opmask=(gdf.lgb=="base")&(gdf.xgb=="base")&(gdf.cat=="base")&(gdf.lin=="base")
op=gdf[opmask]["OOF"].values[0] if opmask.any() else None
print(f"운영(all base) 블렌드 OOF = {op}\n")
print(f"조합 그리드 ({len(gdf)}) 상위 10:")
print(gdf.head(10)[["lgb","xgb","cat","lin","OOF"]].to_string(index=False))
print(f"\n최적: {dict(gdf.iloc[0][['lgb','xgb','cat','lin']])} OOF={gdf.iloc[0]['OOF']:.5f}",f"(운영 대비 {gdf.iloc[0]['OOF']-op:+.5f})" if op else "")
print("최적 가중치:",gdf.iloc[0]["w"])
# 관심 조합 하이라이트
for label,sel in [("cat만 v2",("base","base","v2","base")),("트리 전부 v2",("v2","v2","v2","base")),("선형만 비율",("base","base","base","ratio"))]:
    r=gdf[(gdf.lgb==sel[0])&(gdf.xgb==sel[1])&(gdf.cat==sel[2])&(gdf.lin==sel[3])]
    if len(r): print(f"  [{label}] OOF={r['OOF'].values[0]:.5f}")

운영(all base) 블렌드 OOF = 0.74045

조합 그리드 (16) 상위 10:
 lgb  xgb  cat   lin     OOF
  v2 base   v2 ratio 0.74058
  v2 base   v2  base 0.74057
  v2   v2 base ratio 0.74057
  v2 base base ratio 0.74056
  v2   v2 base  base 0.74056
  v2   v2   v2 ratio 0.74056
  v2   v2   v2  base 0.74055
  v2 base base  base 0.74055
base   v2   v2 ratio 0.74054
base base   v2 ratio 0.74053

최적: {'lgb': 'v2', 'xgb': 'base', 'cat': 'v2', 'lin': 'ratio'} OOF=0.74058 (운영 대비 +0.00013)
최적 가중치: {'lgb': 0.27, 'xgb': 0.2, 'cat': 0.28, 'lin': 0.03, 'nn': 0.22}
  [cat만 v2] OOF=0.74052
  [트리 전부 v2] OOF=0.74055
  [선형만 비율] OOF=0.74046


## 4. §C 최적 조합 → 제출 파일

In [17]:
best=gdf.iloc[0]; sel={"lgb":best["lgb"],"xgb":best["xgb"],"cat":best["cat"],"lin":best["lin"]}
need_tst=[f"lgb_{sel['lgb']}",f"xgb_{sel['xgb']}",f"cat_{sel['cat']}",f"lin_{sel['lin']}"]+([NNK] if NNK else [])
miss=[k for k in need_tst if k not in TST]
if miss:
    print(f"⚠️ 최적 조합의 test 예측 없음: {miss} → 제출 불가. 차선책: test 다 있는 상위 조합 탐색.")
    # test 가용 조합 중 최고
    for _,r in gdf.iterrows():
        s={"lgb":r["lgb"],"xgb":r["xgb"],"cat":r["cat"],"lin":r["lin"]}
        nt=[f"lgb_{s['lgb']}",f"xgb_{s['xgb']}",f"cat_{s['cat']}",f"lin_{s['lin']}"]+([NNK] if NNK else [])
        if all(k in TST for k in nt): best=r; sel=s; print(f"→ test 가용 최고 조합: {sel} OOF={r['OOF']:.5f}"); break
d={"lgb":runr(OOF[f"lgb_{sel['lgb']}"]),"xgb":runr(OOF[f"xgb_{sel['xgb']}"]),"cat":runr(OOF[f"cat_{sel['cat']}"]),"lin":runr(OOF[f"lin_{sel['lin']}"])}
if NNK: d["nn"]=runr(OOF[NNK])
w,a=hill(d,y)
tk={"lgb":f"lgb_{sel['lgb']}","xgb":f"xgb_{sel['xgb']}","cat":f"cat_{sel['cat']}","lin":f"lin_{sel['lin']}","nn":NNK}
ptest=sum(w[k]*runr(TST[tk[k]]) for k in w if w[k]>0 and tk[k] in TST)
sp=find("sample_submission.csv")
if sp:
    s=pd.read_csv(sp); pc=[c for c in s.columns if c!=ID_COL][0]; s[pc]=ptest
else:
    s=pd.DataFrame({ID_COL:test[ID_COL] if 'test' in dir() and ID_COL in test else np.arange(len(ptest)),"probability":ptest})
s.to_csv("/kaggle/working/submission_best_combo.csv",index=False); gdf.to_csv("/kaggle/working/combo_grid.csv",index=False)
print(f"\n최적 제출 조합: {sel} | 블렌드 OOF={a:.5f} | 가중치={ {k:round(v,2) for k,v in w.items() if v>0} }")
print("💾 submission_best_combo.csv → LB 제출"); print("💾 combo_grid.csv (전체 조합 기록)")

⚠️ 최적 조합의 test 예측 없음: ['xgb_base'] → 제출 불가. 차선책: test 다 있는 상위 조합 탐색.
→ test 가용 최고 조합: {'lgb': 'v2', 'xgb': 'v2', 'cat': 'v2', 'lin': 'ratio'} OOF=0.74056

최적 제출 조합: {'lgb': 'v2', 'xgb': 'v2', 'cat': 'v2', 'lin': 'ratio'} | 블렌드 OOF=0.74056 | 가중치={'lgb': 0.27, 'xgb': 0.17, 'cat': 0.3, 'lin': 0.03, 'nn': 0.23}
💾 submission_best_combo.csv → LB 제출
💾 combo_grid.csv (전체 조합 기록)
